In [ ]:
%%sql -r monthly_temp_data
SELECT 
    DATE_TRUNC('MONTH', DATE) AS MONTH,
    ROUND(AVG(VALUE), 2) AS AVG_TEMPERATURE_CELSIUS
FROM SNOWFLAKE_PUBLIC_DATA_FREE.PUBLIC_DATA_FREE.NOAA_WEATHER_METRICS_TIMESERIES t
JOIN SNOWFLAKE_PUBLIC_DATA_FREE.PUBLIC_DATA_FREE.NOAA_WEATHER_STATION_INDEX s
    ON t.NOAA_WEATHER_STATION_ID = s.NOAA_WEATHER_STATION_ID
WHERE s.COUNTRY_NAME = 'United States'
    AND t.VARIABLE = 'average_daily_temperature_note_that_tavg_from_source_s_corresponds_to_an_average_of_hourly_readings_for_the_period_ending_at_2400_utc_rather_than_local_midnight_or_other_local_standard_time_according_to_a_specific_met_service_s_protocol_for_sources_other_than_s_tavg_is_computed_in_a_variety_of_ways_including_traditional_fixed_hours_of_the_day_whereas_taxn_is_solely_computed_as_tmax_tmin_2_0'
    AND DATE >= '2024-01-01' AND DATE < '2026-01-01'
GROUP BY DATE_TRUNC('MONTH', DATE)
ORDER BY MONTH

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import pandas as pd

df = monthly_temp_data.copy()
df['MONTH'] = pd.to_datetime(df['MONTH'])
df['YEAR'] = df['MONTH'].dt.year

fig, ax = plt.subplots(figsize=(12, 5))

for year, group in df.groupby('YEAR'):
    ax.plot(group['MONTH'], group['AVG_TEMPERATURE_CELSIUS'], marker='o', linewidth=2, label=str(year))

ax.set_title('Monthly Average Temperature Across US Weather Stations', fontsize=14, fontweight='bold')
ax.set_xlabel('Month')
ax.set_ylabel('Average Temperature (°C)')
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
ax.xaxis.set_major_locator(mdates.MonthLocator())
plt.xticks(rotation=45, ha='right')
ax.legend(title='Year')
ax.grid(True, alpha=0.3)
ax.axhline(y=0, color='gray', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

In [ ]:
%%sql -r stock_prices
SELECT TICKER, DATE, VALUE AS CLOSE_PRICE
FROM SNOWFLAKE_PUBLIC_DATA_FREE.PUBLIC_DATA_FREE.STOCK_PRICE_TIMESERIES
WHERE TICKER IN ('XOM', 'AAPL')
  AND VARIABLE = 'post-market_close'
  AND DATE >= '2024-01-01' AND DATE < '2025-01-01'
ORDER BY TICKER, DATE

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import pandas as pd

df = stock_prices.copy()
df['DATE'] = pd.to_datetime(df['DATE'])

xom = df[df['TICKER'] == 'XOM']
aapl = df[df['TICKER'] == 'AAPL']

fig, ax1 = plt.subplots(figsize=(14, 6))

color_xom = '#1f77b4'
color_aapl = '#ff7f0e'

ax1.plot(xom['DATE'], xom['CLOSE_PRICE'], color=color_xom, linewidth=1.5, label='XOM (ExxonMobil)')
ax1.set_xlabel('Date')
ax1.set_ylabel('XOM Price ($)', color=color_xom)
ax1.tick_params(axis='y', labelcolor=color_xom)

ax2 = ax1.twinx()
ax2.plot(aapl['DATE'], aapl['CLOSE_PRICE'], color=color_aapl, linewidth=1.5, label='AAPL (Apple)')
ax2.set_ylabel('AAPL Price ($)', color=color_aapl)
ax2.tick_params(axis='y', labelcolor=color_aapl)

ax1.xaxis.set_major_formatter(mdates.DateFormatter('%b'))
ax1.xaxis.set_major_locator(mdates.MonthLocator())
plt.xticks(rotation=45)

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper left')

plt.title('XOM vs AAPL Daily Closing Price — 2024', fontsize=14, fontweight='bold')
ax1.grid(True, alpha=0.3)
fig.tight_layout()
plt.show()

In [ ]:
%%sql -r fl_precip
SELECT 
    DATE_TRUNC('MONTH', t.DATE) AS MONTH,
    ROUND(AVG(t.VALUE), 2) AS AVG_PRECIPITATION_MM,
    COUNT(DISTINCT t.NOAA_WEATHER_STATION_ID) AS STATIONS_REPORTING
FROM SNOWFLAKE_PUBLIC_DATA_FREE.PUBLIC_DATA_FREE.NOAA_WEATHER_METRICS_TIMESERIES t
JOIN SNOWFLAKE_PUBLIC_DATA_FREE.PUBLIC_DATA_FREE.NOAA_WEATHER_STATION_INDEX s
    ON t.NOAA_WEATHER_STATION_ID = s.NOAA_WEATHER_STATION_ID
WHERE s.STATE_NAME = 'Florida'
    AND t.VARIABLE_NAME = 'Precipitation'
    AND t.DATE >= '2024-01-01' AND t.DATE < '2025-01-01'
GROUP BY DATE_TRUNC('MONTH', t.DATE)
ORDER BY MONTH

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import pandas as pd

df = fl_precip.copy()
df['MONTH'] = pd.to_datetime(df['MONTH'])
df['MONTH_NUM'] = df['MONTH'].dt.month
df['LABEL'] = df['MONTH'].dt.strftime('%b')
df['IS_HURRICANE'] = df['MONTH_NUM'].between(6, 11)

colors = ['#e74c3c' if h else '#3498db' for h in df['IS_HURRICANE']]
edge_colors = ['#c0392b' if h else '#2980b9' for h in df['IS_HURRICANE']]

fig, ax = plt.subplots(figsize=(12, 6))
bars = ax.bar(df['LABEL'], df['AVG_PRECIPITATION_MM'], color=colors, edgecolor=edge_colors, linewidth=1.2, width=0.7)

for bar, val in zip(bars, df['AVG_PRECIPITATION_MM']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.15, f'{val}', ha='center', va='bottom', fontsize=10, fontweight='bold')

hurricane_patch = mpatches.Patch(color='#e74c3c', label='Hurricane Season (Jun–Nov)')
normal_patch = mpatches.Patch(color='#3498db', label='Non-Hurricane Season')
ax.legend(handles=[hurricane_patch, normal_patch], loc='upper left', fontsize=11)

ax.set_title('Florida Monthly Avg Daily Precipitation (2024)', fontsize=14, fontweight='bold')
ax.set_ylabel('Avg Daily Precipitation (mm)')
ax.set_xlabel('Month')
ax.grid(axis='y', alpha=0.3)
ax.set_ylim(0, max(df['AVG_PRECIPITATION_MM']) * 1.2)
fig.tight_layout()
plt.show()

In [ ]:
%%sql -r weekly_vol
WITH daily_prices AS (
    SELECT TICKER, DATE, VALUE AS CLOSE_PRICE
    FROM SNOWFLAKE_PUBLIC_DATA_FREE.PUBLIC_DATA_FREE.STOCK_PRICE_TIMESERIES
    WHERE TICKER IN ('XOM', 'CVX', 'NEE')
        AND VARIABLE = 'post-market_close'
        AND DATE >= '2024-01-01' AND DATE < '2025-01-01'
),
daily_returns AS (
    SELECT 
        TICKER,
        DATE,
        (CLOSE_PRICE / LAG(CLOSE_PRICE) OVER (PARTITION BY TICKER ORDER BY DATE) - 1) AS DAILY_RETURN
    FROM daily_prices
)
SELECT 
    DATE_TRUNC('WEEK', DATE) AS WEEK_START,
    TICKER,
    ROUND(STDDEV(DAILY_RETURN) * 100, 4) AS VOLATILITY_PCT
FROM daily_returns
WHERE DAILY_RETURN IS NOT NULL
GROUP BY DATE_TRUNC('WEEK', DATE), TICKER
HAVING COUNT(*) >= 3
ORDER BY WEEK_START, TICKER

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import pandas as pd

df = weekly_vol.copy()
df['WEEK_START'] = pd.to_datetime(df['WEEK_START'])

fig, ax = plt.subplots(figsize=(14, 6))

colors = {'XOM': '#1f77b4', 'CVX': '#ff7f0e', 'NEE': '#2ca02c'}
for ticker, group in df.groupby('TICKER'):
    ax.plot(group['WEEK_START'], group['VOLATILITY_PCT'], linewidth=1.5, label=ticker, color=colors[ticker], marker='o', markersize=3)

ax.set_title('Weekly Price Volatility (Std Dev of Daily Returns) — 2024', fontsize=14, fontweight='bold')
ax.set_xlabel('Week')
ax.set_ylabel('Volatility (%)')
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b'))
ax.xaxis.set_major_locator(mdates.MonthLocator())
plt.xticks(rotation=45)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
fig.tight_layout()
plt.show()

In [ ]:
%%sql -r fed_cpi_data
WITH cpi_monthly AS (
    SELECT DATE, VALUE AS CPI_INDEX
    FROM SNOWFLAKE_PUBLIC_DATA_FREE.PUBLIC_DATA_FREE.FINANCIAL_ECONOMIC_INDICATORS_TIMESERIES
    WHERE VARIABLE = 'CUSRSA0SA01982-84.M'
        AND DATE >= '2022-01-01' AND DATE < '2026-01-01'
),
cpi_yoy AS (
    SELECT 
        c1.DATE,
        ROUND((c1.CPI_INDEX / c2.CPI_INDEX - 1) * 100, 2) AS CPI_YOY_PCT
    FROM cpi_monthly c1
    JOIN cpi_monthly c2 ON c2.DATE = DATEADD('YEAR', -1, c1.DATE)
    WHERE c1.DATE >= '2023-01-01'
),
fed_rate AS (
    SELECT DATE, VALUE * 100 AS FED_FUNDS_RATE_PCT
    FROM SNOWFLAKE_PUBLIC_DATA_FREE.PUBLIC_DATA_FREE.FINANCIAL_ECONOMIC_INDICATORS_TIMESERIES
    WHERE VARIABLE = 'H15_RIFSPFF_N.M'
        AND DATE >= '2023-01-01' AND DATE < '2026-01-01'
)
SELECT 
    COALESCE(c.DATE, f.DATE) AS DATE,
    c.CPI_YOY_PCT,
    f.FED_FUNDS_RATE_PCT
FROM cpi_yoy c
FULL OUTER JOIN fed_rate f ON c.DATE = f.DATE
ORDER BY DATE

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import pandas as pd
import numpy as np

df = fed_cpi_data.copy()
df['DATE'] = pd.to_datetime(df['DATE'])

fig, ax = plt.subplots(figsize=(14, 7))

ax.plot(df['DATE'], df['FED_FUNDS_RATE_PCT'], color='#2c3e50', linewidth=2.5, label='Fed Funds Effective Rate', marker='o', markersize=4)
ax.plot(df['DATE'], df['CPI_YOY_PCT'], color='#e74c3c', linewidth=2.5, label='CPI YoY Change', marker='s', markersize=4)

rate_cuts = [
    ('2024-09-30', '50bp cut\n(Sep 2024)'),
    ('2024-11-30', '25bp cut\n(Nov 2024)'),
    ('2024-12-31', '25bp cut\n(Dec 2024)'),
    ('2025-09-30', '25bp cut\n(Sep 2025)'),
    ('2025-11-30', '25bp cut\n(Nov 2025)'),
]

for date_str, label in rate_cuts:
    dt = pd.to_datetime(date_str)
    row = df[df['DATE'] == dt]
    if not row.empty:
        rate_val = row['FED_FUNDS_RATE_PCT'].values[0]
        ax.annotate(label, xy=(dt, rate_val), xytext=(0, 25),
                    textcoords='offset points', ha='center', fontsize=8, fontweight='bold',
                    color='#2c3e50',
                    arrowprops=dict(arrowstyle='->', color='#2c3e50', lw=1.2))

ax.axhline(y=2, color='green', linestyle='--', alpha=0.5, linewidth=1)
ax.text(df['DATE'].iloc[0], 2.15, 'Fed 2% Target', color='green', fontsize=9, alpha=0.7)

ax.fill_between(df['DATE'], df['CPI_YOY_PCT'], df['FED_FUNDS_RATE_PCT'],
                where=df['FED_FUNDS_RATE_PCT'] > df['CPI_YOY_PCT'],
                alpha=0.1, color='#2c3e50', label='Rate > Inflation (restrictive)')

ax.set_title('Federal Funds Rate vs CPI Year-over-Year Change (2023–2025)', fontsize=14, fontweight='bold')
ax.set_xlabel('Date')
ax.set_ylabel('Rate / Change (%)')
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=3))
plt.xticks(rotation=45)
ax.legend(loc='upper right', fontsize=10)
ax.grid(True, alpha=0.3)
ax.set_ylim(0, 7)
fig.tight_layout()
plt.show()

In [ ]:
%%sql -r temp_xom_corr
WITH weekly_temp AS (
    SELECT 
        DATE_TRUNC('WEEK', t.DATE) AS WEEK_START,
        ROUND(AVG(t.VALUE), 2) AS AVG_TEMP_C
    FROM SNOWFLAKE_PUBLIC_DATA_FREE.PUBLIC_DATA_FREE.NOAA_WEATHER_METRICS_TIMESERIES t
    JOIN SNOWFLAKE_PUBLIC_DATA_FREE.PUBLIC_DATA_FREE.NOAA_WEATHER_STATION_INDEX s
        ON t.NOAA_WEATHER_STATION_ID = s.NOAA_WEATHER_STATION_ID
    WHERE s.COUNTRY_NAME = 'United States'
        AND t.VARIABLE_NAME = 'Maximum temperature'
        AND t.DATE >= '2024-01-01' AND t.DATE < '2025-01-01'
    GROUP BY DATE_TRUNC('WEEK', t.DATE)
),
weekly_xom AS (
    SELECT 
        DATE_TRUNC('WEEK', DATE) AS WEEK_START,
        ROUND(AVG(VALUE), 2) AS AVG_CLOSE_PRICE
    FROM SNOWFLAKE_PUBLIC_DATA_FREE.PUBLIC_DATA_FREE.STOCK_PRICE_TIMESERIES
    WHERE TICKER = 'XOM'
        AND VARIABLE = 'post-market_close'
        AND DATE >= '2024-01-01' AND DATE < '2025-01-01'
    GROUP BY DATE_TRUNC('WEEK', DATE)
)
SELECT 
    t.WEEK_START,
    t.AVG_TEMP_C,
    x.AVG_CLOSE_PRICE,
    CORR(t.AVG_TEMP_C, x.AVG_CLOSE_PRICE) OVER () AS PEARSON_CORR
FROM weekly_temp t
JOIN weekly_xom x ON t.WEEK_START = x.WEEK_START
ORDER BY t.WEEK_START

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import pandas as pd

df = temp_xom_corr.copy()
df['WEEK_START'] = pd.to_datetime(df['WEEK_START'])
corr_value = df['PEARSON_CORR'].iloc[0]

fig, ax1 = plt.subplots(figsize=(14, 6))

color_temp = '#e74c3c'
color_xom = '#2c3e50'

ax1.plot(df['WEEK_START'], df['AVG_TEMP_C'], color=color_temp, linewidth=2, label='Avg Max Temp (°C)', marker='o', markersize=3)
ax1.set_xlabel('Week')
ax1.set_ylabel('Avg Max Temperature (°C)', color=color_temp)
ax1.tick_params(axis='y', labelcolor=color_temp)
ax1.fill_between(df['WEEK_START'], df['AVG_TEMP_C'], alpha=0.1, color=color_temp)

ax2 = ax1.twinx()
ax2.plot(df['WEEK_START'], df['AVG_CLOSE_PRICE'], color=color_xom, linewidth=2, label='XOM Avg Close ($)', marker='s', markersize=3)
ax2.set_ylabel('XOM Closing Price ($)', color=color_xom)
ax2.tick_params(axis='y', labelcolor=color_xom)

ax1.xaxis.set_major_formatter(mdates.DateFormatter('%b'))
ax1.xaxis.set_major_locator(mdates.MonthLocator())
plt.xticks(rotation=45)

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper left', fontsize=10)

ax1.set_title(f'Weekly US Avg Max Temperature vs XOM Closing Price (2024)\nPearson Correlation: {corr_value:.4f}', fontsize=14, fontweight='bold')
ax1.grid(True, alpha=0.3)
fig.tight_layout()
plt.show()